# BY-V queued output gathering

Thread 1 captures 30-second windows from the remote BY-V microphone. Thread 2 processes them with one MOSS model on OpenVINO GPU, trims utterance audio from model timestamps, and stores WAV/JSON pairs locally. Timing reports are separate CSV files.

See [README.md](README.md) for installation, output schema, timing details, and recovery behavior. This notebook launches an isolated worker using the shared OpenVINO environment; no model is loaded into the notebook kernel.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import csv
import json
import sys
import uuid

PROJECT_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                     if (path / "speech_to_text").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from within the project.")
if Path(sys.prefix).resolve() != (PROJECT_ROOT / ".venv").resolve():
    raise RuntimeError("Select the project-local .venv kernel.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from speech_to_text.capture import resolve_microphone

# Capture settings. Device indices are PortAudio indices, not ALSA card numbers.
INPUT_DEVICE = None  # Automatically select the only BY-V input.
SAMPLE_RATE = 48000
INPUT_LATENCY_SECONDS = 1.0  # Buffer scheduling and inference-cleanup pauses.
SEGMENT_SECONDS = 30  # Also sets the model session timestamp limit to 0-30 seconds.
MAX_AUDIO_SECONDS = 120  # Use None to run until interrupted or a STOP file is created.
SOURCE_AUDIO = None  # Example: PROJECT_ROOT / "audio/standby/subway-test-02-thai_n_eng.wav"
REPLAY_REALTIME = True

# Inference and shutdown settings.
MODEL_DIR = PROJECT_ROOT / ".tools" / "moss-openvino-fp16"
OV_DEVICE = "GPU"
MAX_NEW_TOKENS = 1024
CPU_THREADS = 1
WORKER_TIMEOUT_SECONDS = 3600
DRAIN_TIMEOUT_SECONDS = 600
OUTPUT_BASE = PROJECT_ROOT / "research_output_gathering" / "output"

print("Project:", PROJECT_ROOT)
print("Source:", SOURCE_AUDIO if SOURCE_AUDIO is not None else f"BY-V device {resolve_microphone(INPUT_DEVICE)}")
print("Local storage:", OUTPUT_BASE)

Project: /mnt/data/apps/MOSS-ASR-XPU
Source: BY-V device 5
Local storage: /mnt/data/apps/MOSS-ASR-XPU/research_output_gathering/output


Run the next cell to begin collection. It stays busy while recording and draining the queue. Interrupt once to request stop and drain, or create `STOP` inside the printed run directory. The last partial window is retained. The default run captures 120 seconds; compilation and queue drain add to wall-clock time.

The run cell refreshes the lightweight gathering module before creating its configuration, so code updates are picked up in an existing kernel. After changing the setup cell, rerun it to update settings; Restart Kernel and Run All also resets imported code and settings.

Model initialization sets the timestamp window from `SEGMENT_SECONDS` (0-30 seconds by default). Each inference prompt also uses the actual file duration, including a shorter final window. Model intervals outside the audio are retained as rejected records; valid clips are saved and later windows continue. A run with rejections reports `success_with_warnings`, not a fully clean success.

In [2]:
# Refresh orchestration after code updates in a long-lived kernel.
import importlib
import speech_to_text.gathering as gathering
gathering = importlib.reload(gathering)

run_dir = OUTPUT_BASE / ("run-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
                         + "-" + uuid.uuid4().hex[:8])
config = gathering.GatheringConfig(
    output_dir=str(run_dir), model_dir=str(MODEL_DIR), ov_device=OV_DEVICE,
    input_device=INPUT_DEVICE, samplerate=SAMPLE_RATE, segment_seconds=SEGMENT_SECONDS,
    input_latency_seconds=INPUT_LATENCY_SECONDS,
    max_audio_seconds=MAX_AUDIO_SECONDS, max_new_tokens=MAX_NEW_TOKENS,
    cpu_threads=CPU_THREADS,
    source_audio=str(SOURCE_AUDIO) if SOURCE_AUDIO is not None else None,
    replay_realtime=REPLAY_REALTIME,
)
print("Run:", run_dir, flush=True)
print("Request stop from another terminal: touch", run_dir / "STOP", flush=True)
try:
    runtime = gathering.gather_isolated(config, timeout_seconds=WORKER_TIMEOUT_SECONDS,
                              drain_timeout_seconds=DRAIN_TIMEOUT_SECONDS)
    print("Processed:", runtime["processed_count"], "windows")
    print("Rejected model intervals:", runtime["rejected_segment_count"])
finally:
    # The worker owns adapter try/finally cleanup; its process exits before return.
    report_path = run_dir / "reports" / "runtime.json"
    if report_path.is_file():
        summary = json.loads(report_path.read_text())
        print("Status:", summary["status"])
        print("OpenVINO GPU memory after unload:", summary.get("ov_memory_after_unload"))
    print("RTF report:", run_dir / "reports" / "segments.csv")

Run: /mnt/data/apps/MOSS-ASR-XPU/research_output_gathering/output/run-20261101T052319Z-782da787
Request stop from another terminal: touch /mnt/data/apps/MOSS-ASR-XPU/research_output_gathering/output/run-20261101T052319Z-782da787/STOP


[transformers] `MossTranscribeDiarizeProcessor` defines `feature_extractor_class = 'AutoFeatureExtractor'`, which is deprecated. Register the correct mapping in `AutoFeatureExtractor` instead.


OpenVINO execution: {'backend': 'openvino', 'ov_device': 'GPU', 'torch_allocator_scope': 'CPU preprocessing only; not OpenVINO GPU memory', 'audio_window_seconds': 30, 'openvino_batches': [], 'ov_version': '2026.4.1-22982-e213a147257-releases/2026/4', 'ov_device_name': 'Intel(R) Graphics (iGPU)', 'ov_memory_before_compile': {'cl_mem': 0, 'unknown': 0, 'usm_device': 0, 'usm_host': 0, 'usm_shared': 0}, 'encoder_execution_devices': ['GPU.0'], 'decoder_execution_devices': ['GPU.0']}
Captured segment 1: 30.000s; queued=1
OpenVINO generated 1/1024 steps
Saved segment 1: 4 clips; rejected=0; RTF=0.201
Captured segment 2: 30.000s; queued=1
OpenVINO generated 1/1024 steps
Saved segment 2: 2 clips; rejected=0; RTF=0.104
Captured segment 3: 30.000s; queued=1
OpenVINO generated 1/1024 steps
OpenVINO generated 128/1024 steps
Saved segment 3: 5 clips; rejected=0; RTF=0.206
Captured segment 4: 30.000s; queued=1
OpenVINO generated 1/1024 steps
OpenVINO generated 128/1024 steps
Saved segment 4: 5 clips

In [3]:
# Inspect timing separately from model output. This cell does not load a model.
with (run_dir / "reports" / "segments.csv").open(newline="", encoding="utf-8") as file:
    timing_rows = list(csv.DictReader(file))
for row in timing_rows:
    print({key: row[key] for key in ("segment_index", "status", "audio_seconds",
                                     "queue_wait_seconds", "inference_seconds",
                                     "trim_and_store_seconds", "rtf")})
print("Paired transcripts and utterance WAVs:", run_dir / "results")

{'segment_index': '1', 'status': 'success', 'audio_seconds': '30.0', 'queue_wait_seconds': '0.0002989350032294169', 'inference_seconds': '5.999398676001874', 'trim_and_store_seconds': '0.03203817799658282', 'rtf': '0.20139566926663974'}
{'segment_index': '2', 'status': 'success', 'audio_seconds': '30.0', 'queue_wait_seconds': '0.00042170500091742724', 'inference_seconds': '3.0772586180028156', 'trim_and_store_seconds': '0.018217498996818904', 'rtf': '0.10352875313328695'}
{'segment_index': '3', 'status': 'success', 'audio_seconds': '30.0', 'queue_wait_seconds': '0.0004571000026771799', 'inference_seconds': '6.135377548001998', 'trim_and_store_seconds': '0.04475186399940867', 'rtf': '0.20639647053346077'}
{'segment_index': '4', 'status': 'success', 'audio_seconds': '30.0', 'queue_wait_seconds': '0.00030156999855535105', 'inference_seconds': '6.746704592995229', 'trim_and_store_seconds': '0.045861436003178824', 'rtf': '0.22678590466675813'}
Paired transcripts and utterance WAVs: /mnt/dat

Optional analysis with pandas (install it into `.venv` with `uv` if needed):

```python
import pandas as pd
metrics = pd.read_csv(run_dir / "reports" / "segments.csv")
completed = metrics.loc[metrics["status"].isin(["success", "partial"])]
display(completed[["segment_index", "queue_wait_seconds", "inference_seconds", "rtf"]])
```

Speaker labels apply independently within each 30-second input window. Estimated UTC capture times and sample-accurate relative offsets are stored with each utterance. An empty parsed result is retained as `no_segments` with its raw transcript. Inspect `reports/runtime.json` for failures and `recordings/` to recover unprocessed inputs.

## Run summary: transcript, timestamped audio and RTF

Run the final cell after collection, including after a failed run with saved reports. It reads the current `run_dir`, shows all reported windows and their RTF, and pairs each transcript with its existing timestamp-trimmed WAV player. RTF is measured per input window; the overall value is duration-weighted over completed windows (success or partial). No model is loaded and no audio is recut.

Rejected model intervals are shown with their original text and times, but have no audio player.


In [4]:
from IPython.display import HTML, display
import importlib
import speech_to_text.gathering_report as gathering_report

gathering_report = importlib.reload(gathering_report)
if "run_dir" not in globals():
    raise RuntimeError("Run collection first, or set run_dir to an existing saved run directory.")
display(HTML(gathering_report.gathering_summary_html(run_dir)))
